## This is a playground for identifying ICD9 codes from concept IDs in OMOP

In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option("display.max_columns", None)

In [2]:

path_concept = 'omop_vocabs/CONCEPT.csv'
df_concept = pd.read_csv(path_concept,sep="\t", dtype=str)
df_concept.head()


,concept_id,concept_name,domain_id,vocabulary_id,concept_class_id,standard_concept,concept_code,valid_start_date,valid_end_date,invalid_reason
0,45756805,Pediatric Cardiology,Provider,ABMS,Physician Specialty,S,OMOP4821938,19700101,20991231,NaN
1,45756804,Pediatric Anesthesiology,Provider,ABMS,Physician Specialty,S,OMOP4821939,19700101,20991231,NaN
2,45756803,Pathology-Anatomic / Pathology-Clinical,Provider,ABMS,Physician Specialty,S,OMOP4821940,19700101,20991231,NaN
3,45756802,Pathology - Pediatric,Provider,ABMS,Physician Specialty,S,OMOP4821941,19700101,20991231,NaN
4,45756801,Pathology - Molecular Genetic,Provider,ABMS,Physician Specialty,S,OMOP4821942,19700101,20991231,NaN


In [3]:
df_icd9 = df_concept[df_concept['vocabulary_id']=='ICD9CM']
df_icd9.head()

,concept_id,concept_name,domain_id,vocabulary_id,concept_class_id,standard_concept,concept_code,valid_start_date,valid_end_date,invalid_reason
1434783,44833999,Open dislocation of radiocarpal (joint),Condition,ICD9CM,5-dig billing code,NaN,833.12,19700101,20991231,NaN
1434784,44827170,Toxic effect of caustic alkalis,Condition,ICD9CM,4-dig billing code,NaN,983.2,19700101,20991231,NaN
1434785,44825805,Septic shock,Condition,ICD9CM,5-dig billing code,NaN,785.52,19700101,20991231,NaN
1434786,44833450,Other and unspecified disorders of the nervous...,Condition,ICD9CM,3-dig nonbill code,NaN,349,19700101,20991231,NaN
1434787,44821879,Hypopyon,Condition,ICD9CM,5-dig billing code,NaN,364.05,19700101,20991231,NaN


In [4]:
df_icd9.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17564 entries, 1434783 to 1452346
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   concept_id        17564 non-null  object
 1   concept_name      17564 non-null  object
 2   domain_id         17564 non-null  object
 3   vocabulary_id     17564 non-null  object
 4   concept_class_id  17564 non-null  object
 5   standard_concept  0 non-null      object
 6   concept_code      17564 non-null  object
 7   valid_start_date  17564 non-null  object
 8   valid_end_date    17564 non-null  object
 9   invalid_reason    12 non-null     object
dtypes: object(10)
memory usage: 1.5+ MB


# Load condition occurance

In [5]:
path_cond_occurence = 'input_data/condition_occurrence.csv'
df_cond_occurence = pd.read_csv(path_cond_occurence,dtype=str)
df_cond_occurence.head()

,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_start_datetime,condition_end_date,condition_end_datetime,condition_type_concept_id,stop_reason,provider_id,visit_occurrence_id,visit_detail_id,condition_source_value,condition_source_concept_id,condition_status_source_value,condition_status_concept_id
0,1,1,28060,2013-07-05,2013-07-05,2013-07-17,2013-07-17,32020,NaN,NaN,2,0,43878008,28060,NaN,0
1,2,2,381316,2013-10-02,2013-10-02,NaN,NaN,32020,NaN,NaN,20,0,230690007,381316,NaN,0
2,3,2,260139,2013-09-05,2013-09-05,2013-09-12,2013-09-12,32020,NaN,NaN,20,0,10509002,260139,NaN,0
3,4,2,40481087,2011-08-15,2011-08-15,2011-08-29,2011-08-29,32020,NaN,NaN,21,0,444814009,40481087,NaN,0
4,5,2,0,1987-09-16,1987-09-16,NaN,NaN,32020,NaN,NaN,8,0,162864005,4060985,NaN,0


In [6]:
path_concept_relationship = 'omop_vocabs/CONCEPT_RELATIONSHIP.csv'
concept_relationship_df = pd.read_csv(path_concept_relationship,sep="\t", dtype=str)
concept_relationship_df.head()

,concept_id_1,concept_id_2,relationship_id,valid_start_date,valid_end_date,invalid_reason
0,21136156,46089063,Mapped from,20220128,20991231,NaN
1,21126348,46148337,Mapped from,20220128,20991231,NaN
2,21175245,46089024,Mapped from,20220128,20991231,NaN
3,21119785,46148349,Mapped from,20220128,20991231,NaN
4,21175281,46148380,Mapped from,20220128,20991231,NaN


In [7]:
# Ensure both keys are the same type
df_cond_occurence["condition_concept_id"] = df_cond_occurence["condition_concept_id"].astype(str)
df_concept["concept_id"] = df_concept["concept_id"].astype(str)

In [8]:
import pandas as pd

# Step 1: Filter df_concept to only ICD-9 concepts
icd9_concepts = df_concept[df_concept['vocabulary_id'] == 'ICD9CM']

# Step 2: Filter concept_relationship_df to only 'Maps to' relationships where concept_id_1 is in ICD-9
icd9_mappings = concept_relationship_df[
    (concept_relationship_df['relationship_id'] == 'Maps to') &
    (concept_relationship_df['concept_id_1'].isin(icd9_concepts['concept_id']))
]

# Step 3: Merge condition occurrences with standard OMOP concepts
merged_df = df_cond_occurence.merge(
    df_concept, 
    left_on='condition_concept_id', 
    right_on='concept_id', 
    how='left'
).rename(columns={'concept_name': 'omop_condition_name'}).drop(columns=['concept_id'])

# Step 4: Merge with the filtered ICD-9 concept relationships
merged_df = merged_df.merge(
    icd9_mappings,  
    left_on='condition_concept_id', 
    right_on='concept_id_2',  
    how='inner'  # Ensures only mapped conditions remain
)

# Debug: Check intermediate merge
# print("After merging with relationships:")
# print(merged_df[['condition_concept_id', 'concept_id_1', 'concept_id_2']].head())

# Step 5: Merge again to get ICD-9 names and codes
merged_df = merged_df.merge(
    icd9_concepts,  
    left_on='concept_id_1', 
    right_on='concept_id', 
    how='left'
)

# Debug: Check if 'concept_code' exists
print("Columns in df_concept:", df_concept.columns)
print("Columns in merged_df after final merge:", merged_df.columns)

# Rename columns
if 'concept_code' in merged_df.columns:
    merged_df = merged_df.rename(columns={'concept_name': 'icd9_condition_name', 'concept_code': 'icd9_code'})
else:
    print("Warning: 'concept_code' column missing in df_concept!")

# Drop unnecessary columns
merged_df = merged_df.drop(columns=['concept_id', 'concept_id_1', 'concept_id_2'], errors='ignore')

# Debug: Check final result before dropping NaNs
# print("Final merged dataframe preview:")
# print(merged_df.head())

# Step 6: Ensure only valid ICD-9 codes remain
if 'icd9_code' in merged_df.columns:
    merged_df = merged_df.dropna(subset=['icd9_code'])
else:
    print("Error: 'icd9_code' column is missing, skipping dropna step.")

# Step 7: Display final results
print(merged_df.head())


Columns in df_concept: Index(['concept_id', 'concept_name', 'domain_id', 'vocabulary_id',
       'concept_class_id', 'standard_concept', 'concept_code',
       'valid_start_date', 'valid_end_date', 'invalid_reason'],
      dtype='object')
Columns in merged_df after final merge: Index(['condition_occurrence_id', 'person_id', 'condition_concept_id',
       'condition_start_date', 'condition_start_datetime',
       'condition_end_date', 'condition_end_datetime',
       'condition_type_concept_id', 'stop_reason', 'provider_id',
       'visit_occurrence_id', 'visit_detail_id', 'condition_source_value',
       'condition_source_concept_id', 'condition_status_source_value',
       'condition_status_concept_id', 'omop_condition_name', 'domain_id_x',
       'vocabulary_id_x', 'concept_class_id_x', 'standard_concept_x',
       'concept_code_x', 'valid_start_date_x', 'valid_end_date_x',
       'invalid_reason_x', 'concept_id_1', 'concept_id_2', 'relationship_id',
       'valid_start_date_y', 'val

In [9]:
# Step 7: Display 
merged_df.head(5)

,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_start_datetime,condition_end_date,condition_end_datetime,condition_type_concept_id,stop_reason,provider_id,visit_occurrence_id,visit_detail_id,condition_source_value,condition_source_concept_id,condition_status_source_value,condition_status_concept_id,omop_condition_name,domain_id_x,vocabulary_id_x,concept_class_id_x,standard_concept_x,concept_code_x,valid_start_date_x,valid_end_date_x,invalid_reason_x,relationship_id,valid_start_date_y,valid_end_date_y,invalid_reason_y,concept_name,domain_id_y,vocabulary_id_y,concept_class_id_y,standard_concept_y,concept_code_y,valid_start_date,valid_end_date,invalid_reason
0,1,1,28060,2013-07-05,2013-07-05,2013-07-17,2013-07-17,32020,NaN,NaN,2,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN
1,52,7,28060,2015-11-23,2015-11-23,2015-12-01,2015-12-01,32020,NaN,NaN,192,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN
2,156,19,28060,2014-01-26,2014-01-26,2014-02-02,2014-02-02,32020,NaN,NaN,570,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN
3,189,23,28060,2017-02-08,2017-02-08,2017-02-16,2017-02-16,32020,NaN,NaN,675,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN
4,217,26,28060,2014-05-06,2014-05-06,2014-05-17,2014-05-17,32020,NaN,NaN,770,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN


# merge with visit occurrence

In [10]:
visit_oocurrence_df = pd.read_csv('input_data/visit_occurrence.csv',dtype=str)
visit_oocurrence_df.head()

,visit_occurrence_id,person_id,visit_concept_id,visit_start_date,visit_start_datetime,visit_end_date,visit_end_datetime,visit_type_concept_id,provider_id,care_site_id,visit_source_value,visit_source_concept_id,admitting_source_concept_id,admitting_source_value,discharge_to_concept_id,discharge_to_source_value,preceding_visit_occurrence_id
0,5,1,9202,2011-08-19,2011-08-19,2011-08-19,2011-08-19,44818517,NaN,NaN,5c9f315e-8df5-41ae-8a6f-6c22bd11699b,0,0,NaN,0,NaN,NaN
1,2,1,9202,2013-07-05,2013-07-05,2013-07-05,2013-07-05,44818517,NaN,NaN,b094397f-03de-47cd-bd8e-ed59e8c04fd7,0,0,NaN,0,NaN,5
2,4,1,9202,2014-08-22,2014-08-22,2014-08-22,2014-08-22,44818517,NaN,NaN,34f61a48-1b1c-4802-94fd-d25b63cd1ef9,0,0,NaN,0,NaN,2
3,1,1,9202,2016-08-26,2016-08-26,2016-08-26,2016-08-26,44818517,NaN,NaN,b6c23bda-e90a-4ab1-be94-e904cd451df7,0,0,NaN,0,NaN,4
4,3,1,9202,2018-08-31,2018-08-31,2018-08-31,2018-08-31,44818517,NaN,NaN,22cb81da-c0d1-4d66-8e6d-82ae2ab2d81f,0,0,NaN,0,NaN,1


In [11]:
# Merge the merged_df with visit_occurrence based on visit_occurrence_id
merged_df_with_visit = merged_df.merge(
    visit_oocurrence_df, 
    on='visit_occurrence_id',  # Merge on visit_occurrence_id
    how='left'  # or 'inner' depending on your requirements
).drop(columns=['person_id_y']).rename(columns={'person_id_x': 'person_id', 'concept_code_y': 'ICD9_CODE', 'concept_name_y': 'visit_concept_name'})



# Display the resulting DataFrame
merged_df_with_visit.head()


,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_start_datetime,condition_end_date,condition_end_datetime,condition_type_concept_id,stop_reason,provider_id_x,visit_occurrence_id,visit_detail_id,condition_source_value,condition_source_concept_id,condition_status_source_value,condition_status_concept_id,omop_condition_name,domain_id_x,vocabulary_id_x,concept_class_id_x,standard_concept_x,concept_code_x,valid_start_date_x,valid_end_date_x,invalid_reason_x,relationship_id,valid_start_date_y,valid_end_date_y,invalid_reason_y,concept_name,domain_id_y,vocabulary_id_y,concept_class_id_y,standard_concept_y,ICD9_CODE,valid_start_date,valid_end_date,invalid_reason,visit_concept_id,visit_start_date,visit_start_datetime,visit_end_date,visit_end_datetime,visit_type_concept_id,provider_id_y,care_site_id,visit_source_value,visit_source_concept_id,admitting_source_concept_id,admitting_source_value,discharge_to_concept_id,discharge_to_source_value,preceding_visit_occurrence_id
0,1,1,28060,2013-07-05,2013-07-05,2013-07-17,2013-07-17,32020,NaN,NaN,2,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN,9202,2013-07-05,2013-07-05,2013-07-05,2013-07-05,44818517,NaN,NaN,b094397f-03de-47cd-bd8e-ed59e8c04fd7,0,0,NaN,0,NaN,5
1,52,7,28060,2015-11-23,2015-11-23,2015-12-01,2015-12-01,32020,NaN,NaN,192,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN,9202,2015-11-23,2015-11-23,2015-11-23,2015-11-23,44818517,NaN,NaN,5e36dda5-96ae-49a2-9934-260675273f3c,0,0,NaN,0,NaN,172
2,156,19,28060,2014-01-26,2014-01-26,2014-02-02,2014-02-02,32020,NaN,NaN,570,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN,9202,2014-01-26,2014-01-26,2014-01-26,2014-01-26,44818517,NaN,NaN,7d04b70a-e13b-4cb3-975c-23c07f2b5f65,0,0,NaN,0,NaN,582
3,189,23,28060,2017-02-08,2017-02-08,2017-02-16,2017-02-16,32020,NaN,NaN,675,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN,9202,2017-02-08,2017-02-08,2017-02-08,2017-02-08,44818517,NaN,NaN,8e3158ef-b184-488b-82cf-0dd530f12640,0,0,NaN,0,NaN,677
4,217,26,28060,2014-05-06,2014-05-06,2014-05-17,2014-05-17,32020,NaN,NaN,770,0,43878008,28060,NaN,0,Streptococcal sore throat,Condition,SNOMED,Disorder,S,43878008,20020131,20991231,NaN,Maps to,19700101,20991231,NaN,Streptococcal sore throat,Condition,ICD9CM,4-dig billing code,NaN,034.0,19700101,20991231,NaN,9202,2014-05-07,2014-05-07,2014-05-07,2014-05-07,44818517,NaN,NaN,1a5d1a5e-6deb-4561-8d9f-dd6e72bfaaff,0,0,NaN,0,NaN,769


In [12]:
# Optional: Sort by person_id and visit_start_date to ensure correct sequencing
merged_df_with_visit_sorted = merged_df_with_visit.sort_values(['visit_start_date','person_id'], ascending=[True, True])
merged_df_with_visit_sorted['ICD9_CODE'] = merged_df_with_visit_sorted['ICD9_CODE'].str.replace('.','')

merged_df_with_visit_sorted[['visit_occurrence_id','person_id','visit_start_date', 'ICD9_CODE']].tail(20)


,visit_occurrence_id,person_id,visit_start_date,ICD9_CODE
4172,637,20,2018-12-30,842
5135,8917,303,2019-01-01,64263
1438,6330,217,2019-01-04,V22
2206,175,7,2019-01-04,84500
2207,175,7,2019-01-04,8450
2208,175,7,2019-01-04,84509
5603,8935,304,2019-01-05,6424
5604,8935,304,2019-01-05,64240
5605,8935,304,2019-01-05,64241
5606,8935,304,2019-01-05,64242


In [102]:
merged_df_with_visit_clean = merged_df_with_visit_sorted.drop(columns=['provider_id_y','domain_id_y','vocabulary_id_x','standard_concept_y',\
    'invalid_reason_y','valid_start_date_x','valid_start_date_y','valid_end_date_x','valid_end_date_y'])\
    .rename(columns={'provider_id_x': 'provider_id',\
                     'domain_id_x':'domain_id',\
                        'vocabulary_id_y':'vocabulary_id',\
                            'standard_concept_x':'standard_concept',\
                                'concept_code_x':'concept_code',\
                                    'invalid_reason_x':'invalid_reason',\
                                        'concept_class_id_x':'concept_class_id',\
                                            'concept_class_id_y':'concept_class_id_code'})

In [103]:
merged_df_with_visit_clean.head()

,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_start_datetime,condition_end_date,condition_end_datetime,condition_type_concept_id,stop_reason,provider_id,visit_occurrence_id,visit_detail_id,condition_source_value,condition_source_concept_id,condition_status_source_value,condition_status_concept_id,omop_condition_name,domain_id,concept_class_id,standard_concept,concept_code,invalid_reason,relationship_id,concept_name,vocabulary_id,concept_class_id_code,ICD9_CODE,valid_start_date,valid_end_date,invalid_reason,visit_concept_id,visit_start_date,visit_start_datetime,visit_end_date,visit_end_datetime,visit_type_concept_id,care_site_id,visit_source_value,visit_source_concept_id,admitting_source_concept_id,admitting_source_value,discharge_to_concept_id,discharge_to_source_value,preceding_visit_occurrence_id
3048,1292,180,257012,1911-01-30,1911-01-30,NaN,NaN,32020,NaN,NaN,5064,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Other chronic sinusitis,ICD9CM,4-dig billing code,4738,19700101,20991231,NaN,9202,1911-01-30,1911-01-30,1911-01-30,1911-01-30,44818517,NaN,edb3ed1d-921d-41a5-942a-e26621d582e3,0,0,NaN,0,NaN,NaN
3049,1292,180,257012,1911-01-30,1911-01-30,NaN,NaN,32020,NaN,NaN,5064,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Unspecified sinusitis (chronic),ICD9CM,4-dig billing code,4739,19700101,20991231,NaN,9202,1911-01-30,1911-01-30,1911-01-30,1911-01-30,44818517,NaN,edb3ed1d-921d-41a5-942a-e26621d582e3,0,0,NaN,0,NaN,NaN
3050,1292,180,257012,1911-01-30,1911-01-30,NaN,NaN,32020,NaN,NaN,5064,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Chronic sinusitis,ICD9CM,3-dig nonbill code,473,19700101,20991231,NaN,9202,1911-01-30,1911-01-30,1911-01-30,1911-01-30,44818517,NaN,edb3ed1d-921d-41a5-942a-e26621d582e3,0,0,NaN,0,NaN,NaN
3303,4249,610,257012,1912-06-04,1912-06-04,NaN,NaN,32020,NaN,NaN,17272,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Other chronic sinusitis,ICD9CM,4-dig billing code,4738,19700101,20991231,NaN,9202,1912-06-04,1912-06-04,1912-06-04,1912-06-04,44818517,NaN,f00b777c-eaf6-4714-bfc3-7dd71db1907b,0,0,NaN,0,NaN,17247
3304,4249,610,257012,1912-06-04,1912-06-04,NaN,NaN,32020,NaN,NaN,17272,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Unspecified sinusitis (chronic),ICD9CM,4-dig billing code,4739,19700101,20991231,NaN,9202,1912-06-04,1912-06-04,1912-06-04,1912-06-04,44818517,NaN,f00b777c-eaf6-4714-bfc3-7dd71db1907b,0,0,NaN,0,NaN,17247


In [104]:
# visit_df = pd.read_csv(admission_file, dtype=str)
merged_df_with_visit_clean['visit_start_datetime'] = pd.to_datetime(merged_df_with_visit_clean['visit_start_datetime'])
merged_df_with_visit_clean = merged_df_with_visit_clean.sort_values('visit_start_datetime').reset_index(drop=True)

In [26]:
merged_df_with_visit_clean.head(5)

,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_start_datetime,condition_end_date,condition_end_datetime,condition_type_concept_id,stop_reason,provider_id,visit_occurrence_id,visit_detail_id,condition_source_value,condition_source_concept_id,condition_status_source_value,condition_status_concept_id,omop_condition_name,domain_id,concept_class_id,standard_concept,concept_code,invalid_reason,relationship_id,concept_name,vocabulary_id,concept_class_id_code,ICD9_CODE,valid_start_date,valid_end_date,invalid_reason,visit_concept_id,visit_start_date,visit_start_datetime,visit_end_date,visit_end_datetime,visit_type_concept_id,care_site_id,visit_source_value,visit_source_concept_id,admitting_source_concept_id,admitting_source_value,discharge_to_concept_id,discharge_to_source_value,preceding_visit_occurrence_id
0,1292,180,257012,1911-01-30,1911-01-30,NaN,NaN,32020,NaN,NaN,5064,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Other chronic sinusitis,ICD9CM,4-dig billing code,4738,19700101,20991231,NaN,9202,1911-01-30,1911-01-30,1911-01-30,1911-01-30,44818517,NaN,edb3ed1d-921d-41a5-942a-e26621d582e3,0,0,NaN,0,NaN,NaN
1,1292,180,257012,1911-01-30,1911-01-30,NaN,NaN,32020,NaN,NaN,5064,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Unspecified sinusitis (chronic),ICD9CM,4-dig billing code,4739,19700101,20991231,NaN,9202,1911-01-30,1911-01-30,1911-01-30,1911-01-30,44818517,NaN,edb3ed1d-921d-41a5-942a-e26621d582e3,0,0,NaN,0,NaN,NaN
2,1292,180,257012,1911-01-30,1911-01-30,NaN,NaN,32020,NaN,NaN,5064,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Chronic sinusitis,ICD9CM,3-dig nonbill code,473,19700101,20991231,NaN,9202,1911-01-30,1911-01-30,1911-01-30,1911-01-30,44818517,NaN,edb3ed1d-921d-41a5-942a-e26621d582e3,0,0,NaN,0,NaN,NaN
3,4249,610,257012,1912-06-04,1912-06-04,NaN,NaN,32020,NaN,NaN,17272,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Other chronic sinusitis,ICD9CM,4-dig billing code,4738,19700101,20991231,NaN,9202,1912-06-04,1912-06-04,1912-06-04,1912-06-04,44818517,NaN,f00b777c-eaf6-4714-bfc3-7dd71db1907b,0,0,NaN,0,NaN,17247
4,4249,610,257012,1912-06-04,1912-06-04,NaN,NaN,32020,NaN,NaN,17272,0,40055000,257012,NaN,0,Chronic sinusitis,Condition,Disorder,S,40055000,NaN,Maps to,Unspecified sinusitis (chronic),ICD9CM,4-dig billing code,4739,19700101,20991231,NaN,9202,1912-06-04,1912-06-04,1912-06-04,1912-06-04,44818517,NaN,f00b777c-eaf6-4714-bfc3-7dd71db1907b,0,0,NaN,0,NaN,17247


In [17]:
merged_df_with_visit_clean[['ICD9_CODE']].head()

,ICD9_CODE
0,4738
1,4739
2,473
3,4738
4,4739


In [18]:
# merged_df_with_visit_clean.to_csv('synthea1k_ICD9.csv',index=True)

In [105]:


# Read the CSV files into DataFrames


query_drug_exposure = "input_data/drug_exposure.csv"
query_measurement = "input_data/measurement.csv"
query_procedure_occurrence = "input_data/procedure_occurrence.csv"
query_person = "input_data/person.csv"

df_drug_exposure = pd.read_csv(query_drug_exposure, dtype=str)
df_measurement = pd.read_csv(query_measurement, dtype=str)
df_procedure_occurrence = pd.read_csv(query_procedure_occurrence, dtype=str)
df_person = pd.read_csv(query_person, dtype=str)



In [ ]:
df_drug_exposure.head(5)

In [ ]:
df_drug_exposure = df_drug_exposure.merge(df_concept[['concept_id', 'concept_name']], left_on='drug_concept_id', right_on='concept_id', how='left').rename(columns={'concept_name': 'drug_name'}).drop(columns=['concept_id'])
df_drug_exposure.head(5)

In [ ]:
df_drug_exposure = df_drug_exposure.merge(df_concept[['concept_id', 'concept_name']], left_on='drug_concept_id', right_on='concept_id', how='left').rename(columns={'concept_name': 'drug_name'}).drop(columns=['concept_id'])

df_measurement = df_measurement.merge(df_concept[['concept_id', 'concept_name']], left_on='measurement_concept_id', right_on='concept_id', how='left').rename(columns={'concept_name': 'measurement_name'}).drop(columns=['concept_id'])

df_procedure_occurrence = df_procedure_occurrence.merge(df_concept[['concept_id', 'concept_name']], left_on='procedure_concept_id', right_on='concept_id', how='left').rename(columns={'concept_name': 'procedure_name'}).drop(columns=['concept_id'])


In [28]:
df_person.head(10)

,person_id,gender_concept_id,year_of_birth,month_of_birth,day_of_birth,birth_datetime,race_concept_id,ethnicity_concept_id,location_id,provider_id,care_site_id,person_source_value,gender_source_value,gender_source_concept_id,race_source_value,race_source_concept_id,ethnicity_source_value,ethnicity_source_concept_id,gender
0,1,8507,1974,12,6,1974-12-06,8527,0,NaN,NaN,NaN,0033e4bc-e630-47be-8c89-fcd332cef687,M,0,white,0,italian,0,MALE
1,2,8532,1950,8,2,1950-08-02,8527,0,NaN,NaN,NaN,0062fa0d-3ae2-4f99-93b0-e404e062e4f5,F,0,white,0,irish,0,FEMALE
2,3,8507,1910,12,30,1910-12-30,8527,0,NaN,NaN,NaN,0088e7df-562f-462a-b480-d2e19e77b984,M,0,white,0,irish,0,MALE
3,4,8532,1959,5,23,1959-05-23,8527,0,NaN,NaN,NaN,008d249f-31a0-4258-a04f-5f737e7b6791,F,0,white,0,german,0,FEMALE
4,5,8507,1963,10,19,1963-10-19,8527,0,NaN,NaN,NaN,00b3de6e-af6d-4534-856f-66309cc9fbd0,M,0,white,0,italian,0,MALE
5,6,8532,1980,3,23,1980-03-23,8527,0,NaN,NaN,NaN,00b6eeee-0aa7-4266-89ca-4f41cc7ca65c,F,0,white,0,french,0,FEMALE
6,7,8507,2013,1,5,2013-01-05,8527,0,NaN,NaN,NaN,00caf4e3-1e3d-4515-87f9-712fe8bed057,M,0,white,0,irish,0,MALE
7,8,8507,1970,11,14,1970-11-14,8527,0,NaN,NaN,NaN,00e740a5-3cc1-41fc-8456-fd7ca8c0b901,M,0,white,0,italian,0,MALE
8,9,8507,2014,9,17,2014-09-17,8527,0,NaN,NaN,NaN,00eda281-ff32-44d0-91c1-55318818cd72,M,0,white,0,irish,0,MALE
9,10,8507,1949,6,4,1949-06-04,8527,0,NaN,NaN,NaN,00fb60a4-3074-46db-a407-aaa7cb48ca0a,M,0,white,0,irish,0,MALE


In [106]:
df_person = df_person.merge(df_concept[['concept_id', 'concept_name']], left_on='gender_concept_id', right_on='concept_id', how='left').rename(columns={'concept_name': 'gender'}).drop(columns=['concept_id'])
df_person = df_person.merge(df_concept[['concept_id', 'concept_name']], left_on='race_concept_id', right_on='concept_id', how='left').rename(columns={'concept_name': 'race'}).drop(columns=['concept_id'])


In [107]:
df_person.head(5)

,person_id,gender_concept_id,year_of_birth,month_of_birth,day_of_birth,birth_datetime,race_concept_id,ethnicity_concept_id,location_id,provider_id,care_site_id,person_source_value,gender_source_value,gender_source_concept_id,race_source_value,race_source_concept_id,ethnicity_source_value,ethnicity_source_concept_id,gender,race
0,1,8507,1974,12,6,1974-12-06,8527,0,NaN,NaN,NaN,0033e4bc-e630-47be-8c89-fcd332cef687,M,0,white,0,italian,0,MALE,White
1,2,8532,1950,8,2,1950-08-02,8527,0,NaN,NaN,NaN,0062fa0d-3ae2-4f99-93b0-e404e062e4f5,F,0,white,0,irish,0,FEMALE,White
2,3,8507,1910,12,30,1910-12-30,8527,0,NaN,NaN,NaN,0088e7df-562f-462a-b480-d2e19e77b984,M,0,white,0,irish,0,MALE,White
3,4,8532,1959,5,23,1959-05-23,8527,0,NaN,NaN,NaN,008d249f-31a0-4258-a04f-5f737e7b6791,F,0,white,0,german,0,FEMALE,White
4,5,8507,1963,10,19,1963-10-19,8527,0,NaN,NaN,NaN,00b3de6e-af6d-4534-856f-66309cc9fbd0,M,0,white,0,italian,0,MALE,White


In [108]:
merged_df_with_visit_clean_person = merged_df_with_visit_clean.merge(df_person[['person_id','gender_concept_id', 'gender', 'year_of_birth','birth_datetime','race_concept_id','race']], on='person_id', how='left')

In [109]:
# Check for invalid or missing values in the relevant columns
merged_df_with_visit_clean_person['visit_start_datetime'] = pd.to_datetime(
    merged_df_with_visit_clean_person['visit_start_datetime'], errors='coerce'
)
merged_df_with_visit_clean_person['year_of_birth'] = pd.to_numeric(
    merged_df_with_visit_clean_person['year_of_birth'], errors='coerce'
)

# Drop rows with missing or invalid values in 'visit_start_datetime' or 'year_of_birth'
merged_df_with_visit_clean_person = merged_df_with_visit_clean_person.dropna(
    subset=['visit_start_datetime', 'year_of_birth']
)

# Calculate age directly as an integer
merged_df_with_visit_clean_person['age'] = (
    merged_df_with_visit_clean_person['visit_start_datetime'].dt.year - 
    merged_df_with_visit_clean_person['year_of_birth'].astype(int)
)

# Check for NaN values in the 'age' column
nan_age_count = merged_df_with_visit_clean_person['age'].isna().sum()
print(f"Number of NaN values in 'age' column after cleaning: {nan_age_count}")

# Optionally, fill NaN values in 'age' with a default value (e.g., 0)
merged_df_with_visit_clean_person['age'] = merged_df_with_visit_clean_person['age'].fillna(-1).astype(int)

# Display the first 5 rows of the 'age' column
print(merged_df_with_visit_clean_person[['age']].head(5))

Number of NaN values in 'age' column after cleaning: 0
   age
0    3
1    3
2    3
3    4
4    4


In [81]:
merged_df_with_visit_clean_person[['person_id','visit_start_date','birth_datetime','age']].head(5)

,person_id,visit_start_date,birth_datetime,age
0,180,1911-01-30,1908-08-17,3
1,180,1911-01-30,1908-08-17,3
2,180,1911-01-30,1908-08-17,3
3,610,1912-06-04,1908-08-17,4
4,610,1912-06-04,1908-08-17,4


In [91]:
# Filter rows where the 'age' column is equal to -1 and display the first 5 rows
merged_df_with_visit_clean_person[merged_df_with_visit_clean_person['age'] == -1].head(5)

,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_start_datetime,condition_end_date,condition_end_datetime,condition_type_concept_id,stop_reason,provider_id,visit_occurrence_id,visit_detail_id,condition_source_value,condition_source_concept_id,condition_status_source_value,condition_status_concept_id,omop_condition_name,domain_id,concept_class_id,standard_concept,concept_code,invalid_reason,relationship_id,concept_name,vocabulary_id,concept_class_id_code,ICD9_CODE,valid_start_date,valid_end_date,invalid_reason,visit_concept_id,visit_start_date,visit_start_datetime,visit_end_date,visit_end_datetime,visit_type_concept_id,care_site_id,visit_source_value,visit_source_concept_id,admitting_source_concept_id,admitting_source_value,discharge_to_concept_id,discharge_to_source_value,preceding_visit_occurrence_id,gender_concept_id,gender,gender,gender,year_of_birth,birth_datetime,race_concept_id,race,race,race,age


In [115]:
# Ensure 'race_concept_id' is numeric and replace NaN with 0
merged_df_with_visit_clean_person['race_concept_id'] = pd.to_numeric(
    merged_df_with_visit_clean_person['race_concept_id'], errors='coerce'
).fillna(-1)

# Filter rows where 'race_concept_id' is not equal to 0
filtered_df = merged_df_with_visit_clean_person[merged_df_with_visit_clean_person['race_concept_id'] != 0]

# Display the first 5 rows of the filtered DataFrame
filtered_df[['race_concept_id', 'race']].head(5)

,race_concept_id,race
13,8515,Asian
14,8515,Asian
15,8515,Asian
16,8515,Asian
17,8515,Asian


In [110]:
merged_df_with_visit_clean_person[merged_df_with_visit_clean_person['race_concept_id']!=0][['race_concept_id','race']].head(5)

,race_concept_id,race
0,0,No matching concept
1,0,No matching concept
2,0,No matching concept
3,0,No matching concept
4,0,No matching concept


In [114]:
filtered = merged_df_with_visit_clean_person[merged_df_with_visit_clean_person['race_concept_id'] != 0]
filtered[['race_concept_id', 'race']].head(15)


,race_concept_id,race
13,8515,Asian
14,8515,Asian
15,8515,Asian
16,8515,Asian
17,8515,Asian
18,8527,White
19,8527,White
20,8527,White
21,8527,White
22,8527,White


In [ ]:
merged_df_with_all = merged_df_with_visit.merge(df_drug_exposure[['person_id', 'drug_name', 'drug_exposure_start_date','visit_occurrence_id']], on='visit_occurrence_id', how='left')

merged_df_with_all.head(3)

In [ ]:
df_measurement.head(3)

In [ ]:
merged_df_with_all = merged_df_with_all.merge(df_measurement[['person_id', 'measurement_name','measurement_date', 'value_as_number', 'unit_concept_id', 'visit_occurrence_id']], on='visit_occurrence_id', how='left')


In [ ]:
merged_df_with_all[['condition_start_date','drug_exposure_start_date','measurement_date']].head(5)

In [ ]:
df_procedure_occurrence.head(3)

In [ ]:
merged_df_with_all2 = merged_df_with_all.merge(df_procedure_occurrence[[ 'procedure_name', 'procedure_date','visit_occurrence_id']], on='visit_occurrence_id', how='left')


In [ ]:
merged_df_with_all.head(5)

In [ ]:
merged_df_with_visit_clean.shape

In [ ]:
from sqlalchemy import create_engine

try:
    db_engine = create_engine(f"postgresql+psycopg2://postgres:@localhost:5432/hisgt_db")
    print("Database connection established successfully!")
except Exception as e:
    print(f"Error connecting to the database: {e}")


In [ ]:

import pandas as pd

query_concept = "SELECT * FROM \"CONCEPT\";"

        # Load data from Database
df_concept = pd.read_sql(query_concept, db_engine)

In [ ]:
from preprocessing import OMOP_to_ICD9_conversion
import yaml
import os
from pathlib import Path

# filepath = Path(__file__).parents[1]
config_file_path = os.path.join("/Users/uqhkamel/dev/HiSGT_forked/HiSGT", "config.yml")
print(config_file_path)


with open(config_file_path) as config_file:
    config = yaml.safe_load(config_file)
    print(config)

visit_df = OMOP_to_ICD9_conversion(db_name="postgres", db_config=config)
visit_df.head(5)